# From geodetic coordinates to a local ENU frame

This lab connects longitude/latitude/ellipsoidal height, ECEF XYZ and local East-North-Up coordinates. It uses WGS 84 by default and reproduces an EPSG:9836 numerical example. Read the [method lesson](../algorithms/geodesy/ecef-enu.md) and [implementation contract](../implementations/python/ecef-enu/README.md).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

current = Path.cwd().resolve()
repo_root = next((p for p in (current, *current.parents)
                  if (p / 'implementations' / 'python' / 'ecef-enu').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Run this notebook from a checkout of geo-dev-hub/spatial-analysis.')
package = repo_root / 'implementations' / 'python' / 'ecef-enu'
sys.path.insert(0, str(package))
from ecef_enu import geodetic_to_ecef, ecef_to_enu, enu_to_ecef, geodetic_to_enu


## 1. Convert geodetic coordinates to ECEF

Longitude and latitude are in degrees; height is ellipsoidal metres. The output columns are X, Y and Z in metres.

In [ ]:
lon, lat, height = 5.0, 55.0, 200.0
origin_xyz = geodetic_to_ecef(lon, lat, height)
print('Origin ECEF XYZ (m):', origin_xyz)
print('Equator/prime-meridian point:', geodetic_to_ecef(0, 0, 0))


## 2. Convert the published ECEF point to ENU

The origin and target are taken from EPSG method 9836. This checks the signs and order of the local axes.

In [ ]:
target_xyz = np.array([3771793.968, 140253.342, 5124304.349])
target_enu = ecef_to_enu(target_xyz, origin_longitude_deg=lon,
                         origin_latitude_deg=lat,
                         origin_ellipsoidal_height_m=height)
print('East, North, Up (m):', target_enu)
print('Reference: approximately [-189013.869, -128642.040, -4220.171]')


## 3. Inspect the local axes

The arrows below show the local ENU unit directions expressed in the global ECEF frame. The origin is the reference point. The drawing is schematic: the arrows are scaled for visibility, not to Earth radius.

In [ ]:
lam, phi = np.deg2rad(lon), np.deg2rad(lat)
east_axis = np.array([-np.sin(lam), np.cos(lam), 0.0])
north_axis = np.array([-np.sin(phi)*np.cos(lam), -np.sin(phi)*np.sin(lam), np.cos(phi)])
up_axis = np.array([np.cos(phi)*np.cos(lam), np.cos(phi)*np.sin(lam), np.sin(phi)])
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
scale = 1.0
for axis, label in [(east_axis, 'East'), (north_axis, 'North'), (up_axis, 'Up')]:
    ax.quiver(0, 0, 0, *(axis*scale), length=1, normalize=True, label=label)
ax.scatter([0], [0], [0], marker='o', label='Local origin')
ax.set_xlabel('ECEF X direction'); ax.set_ylabel('ECEF Y direction'); ax.set_zlabel('ECEF Z direction')
ax.set_title('Local ENU axes expressed in ECEF')
ax.legend(); plt.show()


## 4. Check the inverse transformation

The ENU-to-ECEF operation uses the transpose of the rotation matrix and adds the ECEF origin. This should recover the original XYZ point up to floating-point rounding.

In [ ]:
restored_xyz = enu_to_ecef(target_enu, origin_longitude_deg=lon,
                           origin_latitude_deg=lat,
                           origin_ellipsoidal_height_m=height)
print('Restored XYZ:', restored_xyz)
print('Maximum absolute round-trip error (m):', np.max(np.abs(restored_xyz-target_xyz)))


## 5. Compare nearby points in the local frame

A small longitude change at this latitude produces a mostly eastward displacement. The local ENU values are expressed in metres and depend on the chosen origin.

In [ ]:
nearby = geodetic_to_enu([5.0, 5.001, 5.0], [55.0, 55.0, 55.001],
                         [200.0, 200.0, 200.0],
                         origin_longitude_deg=5.0,
                         origin_latitude_deg=55.0,
                         origin_ellipsoidal_height_m=200.0)
for label, row in zip(['Origin', 'Eastward move', 'Northward move'], nearby):
    print(f'{label:16s}: E={row[0]:9.3f} m, N={row[1]:9.3f} m, U={row[2]:8.3f} m')


## Investigate

1. Why must the origin be converted to ECEF before rotating the target difference?
2. Why does ellipsoidal height matter?
3. What does the Up axis represent at the local origin?
4. Why is the ENU frame not a global map projection?
5. What changes when the local origin moves?

Complete the [ECEF/ENU exercise](../exercises/ecef-enu.md).